# Read Data

In [0]:
from pyspark.sql.types import * 
from pyspark.sql.functions import * 

## Read Data in CSV file format

In [0]:
df = spark.read.format('csv').option('inferSchema', True).option('header', True).load('/Volumes/workspace/default/tutorial1/BigMart Sales Data.csv')

In [0]:
df.printSchema()

### Change Schema DDL

In [0]:
my_ddl_schema = '''
                    Item_Identifier string,
                    Item_Weight string,
                    Item_Fat_Content string,
                    Item_Visibility double,
                    Item_Type string,
                    Item_MRP double,
                    Outlet_Identifier string,
                    Outlet_Establishment_Year long,
                    Outlet_Size string,
                    Outlet_Location_Type string,
                    Outlet_Type string,
                    Item_Outlet_Sales double
                '''

In [0]:
df = spark.read.format('csv').schema(my_ddl_schema).option('header', True).load('/Volumes/workspace/default/tutorial1/BigMart Sales Data.csv')

In [0]:
df.printSchema()

## Read Data in JSON format

In [0]:
df_json = spark.read.format('json').option('inferSchema', True).option('header', True).load('/Volumes/workspace/default/tutorial1/Drivers Data.json')

In [0]:
df_json.display()

## Struct Type Schema PySpark SQL

In [0]:
schema_structure = StructType([
                                    StructField('Item_Identifier', StringType(), True),
                                    StructField('Item_Weight', DoubleType(), True),
                                    StructField('Item_Fat_Content', StringType(), True),
                                    StructField('Item_Visibility', DoubleType(), True),
                                    StructField('Item_Type', StringType(), True),
                                    StructField('Item_MRP', DoubleType(), True),
                                    StructField('Outlet_Identifier', StringType(), True),
                                    StructField('Outlet_Establishment_Year', LongType(), True),
                                    StructField('Outlet_Size', StringType(), True),
                                    StructField('Outlet_Location_Type', StringType(), True),
                                    StructField('Outlet_Type', StringType(), True),
                                    StructField('Item_Outlet_Sales', DoubleType(), True)
                                ])

In [0]:
df = spark.read.format('csv').schema(schema_structure).option('header', True).load('/Volumes/workspace/default/tutorial1/BigMart Sales Data.csv')

## SELECT

In [0]:
df.select("Item_Fat_Content", "Item_Type").display()


In [0]:
df.select(col('Item_Fat_Content'), col('Item_Identifier')).display()

## ALIAS

In [0]:
df.select(col('Item_Identifier')).alias('Item_ID').display()

## COUNT

In [0]:
df.select(col('Item_Identifier')).count()

## DISTINCT

In [0]:
df.select(col("Item_Type")).distinct().count()

## FILTER/WHERE

In [0]:
df.filter(col('Item_Fat_Content')== 'Regular').display()

In [0]:
df.filter((col('Item_Fat_Content') == 'Regular') & (col('Item_Weight') > 10)).display()

In [0]:
df.filter((col('Outlet_Location_Type').isin('Tier 3','Tier 2') & col("Outlet_Size").isNull())).display()

## Rename Column

In [0]:
df = df.withColumnsRenamed({'Item_Identifier': 'Item_ID', 'Item_Weight': 'Item_W'})

## WITH COLUMN

Adds a new column


In [0]:
df = df.withColumn('new_col', lit(None))

In [0]:
df.withColumn('multiply', col('Item_Weight')*col('Item_MRP')).display()

In [0]:
df.withColumn('Item_Fat_Content', regexp_replace(col('Item_Fat_Content'),'Low Fat', 'LF'))\
    .withColumn('Item_Fat_Content', regexp_replace(col('Item_Fat_Content'),'Regular', 'Reg'))\
    .display()


## Type Casting


In [0]:
df = df.withColumn('Item_Weight', col("Item_Weight").cast(StringType()))

## Sort and Order BY

In [0]:
df.filter(col('Item_Weight').isNotNull()).sort(col('Item_Weight').desc()).display()

In [0]:
df.sort(['Item_Weight', 'Item_MRP'], ascending = [0,0]).display()

In [0]:
df.sort(['Item_Weight', 'Item_MRP'], ascending = [0,1]).display()

In [0]:
df.limit(10).display()

## Drop

In [0]:
df.drop(col('Item_Weight')).display()

## Drop Duplicates

In [0]:
df.dropDuplicates().display()

In [0]:
df.drop_duplicates(subset=['Item_Type']).display()

In [0]:
df.distinct().display()

## Union and Union by name

In [0]:
data_1 = [(1, "Cam"), (2, "Tom")]
schema_1 = 'id INT, name STRING'
df_1 = spark.createDataFrame(data_1, schema_1)

data_2 = [(3, "Dean"), (4, "James")]
schema_2 = 'id INT, name STRING'
df_2 = spark.createDataFrame(data_2, schema_2)


In [0]:
df_1.withColumn("id", col("id").cast(StringType()))
df_2.withColumn("id", col("id").cast(StringType()))

In [0]:
df_1.union(df_2).display()

In [0]:
df_1.unionByName(df_2).display()

In [0]:
df.select(lower(col("Item_Type"))).display()
df.select(upper(col("Item_Type"))).display()
df.select(initcap(col("Item_Type"))).display()

## DATE FUNCTIONS


In [0]:
from pyspark.sql.types import * 
from pyspark.sql.functions import * 

In [0]:
df = spark.read.format('csv').option('inferSchema', True).option('header', True).load('/Volumes/workspace/default/tutorial1/BigMart Sales Data.csv')

In [0]:
df = df.withColumn("curr_date", current_date())

In [0]:
df = df.withColumn("curr_timestamp", current_timestamp())

In [0]:
df = df.withColumn("week_after", date_add('curr_date', 7))
df = df.withColumn("week_before", date_sub(col('curr_date'), 7))
df = df.withColumn("days_between", datediff(col('curr_date'), col('week_before')))
df.display()

### Date Format

In [0]:
df = df.withColumn('week_after', date_format('week_after', 'mm-DD-yyyy'))
df.display()


## Handle NULL

In [0]:
df.dropna('all').display()

In [0]:
df.dropna('any').display()

In [0]:
df.dropna(subset = ['Item_Weight']).display()

In [0]:
df.fillna(0).display()



## SLIPT and INDEX

In [0]:
df.withColumn('Outlet_Type_Split', explode(split(col('Outlet_Type'), ' '))).display()

In [0]:
df.withColumn('Type1_flag', col('Outlet_Type').contains('Type1')).display()

In [0]:
df.groupBy("Item_Type").agg(min('Item_MRP').alias('min_item_mrp')).display()

In [0]:
df.groupBy("Item_Type", "Outlet_Size").agg(min('Item_MRP').alias('min_item_mrp'), avg("Item_MRP").alias('avg_item_mrp')).display()

In [0]:
df.groupBy('Item_Identifier').agg(collect_list('Item_Type').alias('Item_Type')).display()

In [0]:
df.groupBy('Item_Type').pivot('Outlet_Size').agg(avg('Item_MRP')).alias('avg_item_mrp').display()

In [0]:
df.withColumn('veggie_flag', when(col('Item_Type')=='Meat', 'not veggie').otherwise('veggie')).display()

In [0]:
df.withColumn('veggie_flag', when((col('Item_Type')=='Meat') & (col("Outlet_Size") == 'High'), 'not veggie').otherwise('veggie')).display()

In [0]:
df.withColumn('Veggies', when(((col('Item_Type')=='Meat') & (col("Item_MRP") < 100)), 'not viggie not expensive')\
                        .when((col('Item_Type')=='Meat') & (col("Item_MRP") > 100), 'not viggie very expensive').otherwise('food')).display()

## JOINS

### INNER JOIN

In [0]:
data1 = [(1, 'a', 'vocal'), (2, 'b', 'consonant'), (3, 'c', 'consonant'), (4, 'd', 'consonant'), (5, 'e', 'vocal')]
df1 = spark.createDataFrame(data1, ['position', 'letter', 'type'])
data2 = [(6, 'f', 'consonant'), (7, 'g', 'consonant'), (8, 'h', 'consonant'), (9, 'i', 'vocal'), (10, 'j', 'consonant'), (11, 'k', 'consonant'), (12, 'l', 'consonant'), (13, 'm', 'consonant'), (14, 'n', 'consonant'), (15, 'o', 'vocal')]
df2 = spark.createDataFrame(data2, ['position', 'letter', 'type'])

In [0]:
df1.join(df2, df1['letter'] == df2['letter'], 'inner').display()
df1.join(df2, df1['letter'] == df2['letter'], 'left').display()
df1.join(df2, df1['letter'] == df2['letter'], 'right').display()
df1.join(df2, df1['letter'] == df2['letter'], 'outer').display()
df1.join(df2, df1['letter'] == df2['letter'], 'leftanti').display()
df1.join(df2, df1['letter'] == df2['letter'], 'leftsemi').display()
df1.join(df2, df1['letter'] == df2['letter'], 'anti').display()
